This vignette demonstrates how to use the bioc2ri ecosystem to seamlessly move genomic and single-cell data between Python and R.

## Environment Setup
First, we initialize our conversion engines. In this architecture, plugins are designed to be modular. While you can combine them, the bioc_experiment_plugin is smart enough to call the bioc_ranges_plugin and biocpy_plugin internally as needed.

In [1]:
import numpy as np
from biocframe import BiocFrame
from iranges import IRanges
from genomicranges import GenomicRanges
from singlecellexperiment import SingleCellExperiment

# Import our plugins
from bioc2ri import base_plugin
from bioc2ri.bioc_ranges_plugin import bioc_ranges_plugin
from bioc2ri.summarizedexperiment_plugin import bioc_experiment_plugin

# Initialize the standalone experiment engine
# It will internally resolve ranges and frames.
exp_engine = bioc_experiment_plugin()
ranges_engine = bioc_ranges_plugin()

## Constructing Genomic Ranges
We'll start by creating a GenomicRanges object in Python. This represents the genomic coordinates of our features (e.g., genes or peaks).

In [2]:
# Create 5 genomic features
gr = GenomicRanges(
    seqnames=["chr1", "chr1", "chr2", "chrX", "chrY"],
    ranges=IRanges(start=[100, 200, 300, 400, 500], width=[50, 50, 50, 50, 50]),
    strand=["+", "-", "+", "*", "-"],
    mcols=BiocFrame({"gene_id": ["G1", "G2", "G3", "G4", "G5"]})
)

# Python -> R Conversion
r_gr = ranges_engine.py2r(gr)
print(f"R GRanges type: {type(r_gr)}")

R GRanges type: <class 'rpy2.robjects.methods.RS4'>


In [3]:
from genomicranges import GenomicRanges, CompressedGenomicRangesList
from compressed_lists import Partitioning
from rpy2.robjects import r
    
# 1. In Python: One Gene, Two Exons
exon1 = GenomicRanges(seqnames=["chr1"], ranges=IRanges([100], [50]), strand=["+"])
exon2 = GenomicRanges(seqnames=["chr1"], ranges=IRanges([200], [50]), strand=["+"])

grl = CompressedGenomicRangesList.from_list(
    lst=[exon1, exon2], 
    names=["Gene_A", "Gene_B"] # For simplicity, each gene has 1 exon here
)

# 2. Convert to R
# Internal: Converts to a single flat GRanges + Partitioning ends [1, 2]
r_grl = ranges_engine.py2r(grl)

# 3. Verify in R
print(r["class"](r_grl)) # 'CompressedGRangesList'
print(r["elementNROWS"](r_grl)) # 1 1

# 4. Round-trip
grl_back = ranges_engine.r2py(r_grl)
assert isinstance(grl_back, CompressedGenomicRangesList)

[1] "CompressedGRangesList"
attr(,"package")
[1] "GenomicRanges"

Gene_A Gene_B 
     1      1 



## Building a SingleCellExperiment
Now, let's wrap those ranges into a SingleCellExperiment (SCE). This is the "boss" of Bioconductor objects, containing assays (matrices), row/column metadata, and reduced dimensions.

In [4]:
# 1. Setup Assays (5 genes x 4 cells)
counts = np.random.poisson(lam=5, size=(5, 4))
logcounts = np.log1p(counts)

# 2. Setup Column Metadata (Samples)
coldata = BiocFrame({
    "sample": ["S1", "S1", "S2", "S2"],
    "batch": ["A", "A", "B", "B"]
}, row_names=["Cell_1", "Cell_2", "Cell_3", "Cell_4"])

# 3. Create the SCE
sce = SingleCellExperiment(
    assays={"counts": counts, "logcounts": logcounts},
    row_ranges=gr, # Using the GRanges we made above
    column_data=coldata,
    reduced_dims={"PCA": np.random.normal(size=(4, 2))}
)

# Python -> R Conversion
r_sce = exp_engine.py2r(sce)

## Verification in R
Using rpy2, we can verify that the R-side SingleCellExperiment is fully valid and parallel.

In [5]:
from rpy2.robjects import r

# Check R class
print(r["class"](r_sce)) # Should show ['SingleCellExperiment', 'RangedSummarizedExperiment', ...]

# Check dimensions
print(f"R Dimensions: {r['dim'](r_sce)}")

# Check if rowRanges transitioned correctly
print(f"R Strand info: {r['strand'](r['rowRanges'](r_sce))}")

[1] "SingleCellExperiment"
attr(,"package")
[1] "SingleCellExperiment"

R Dimensions: [1] 5 4

R Strand info: RleList of length 5
[[1]]
factor-Rle of length 0 with 0 runs
  Lengths: 
  Values : 
Levels(3): + - *

[[2]]
factor-Rle of length 0 with 0 runs
  Lengths: 
  Values : 
Levels(3): + - *

[[3]]
factor-Rle of length 0 with 0 runs
  Lengths: 
  Values : 
Levels(3): + - *

[[4]]
factor-Rle of length 0 with 0 runs
  Lengths: 
  Values : 
Levels(3): + - *

[[5]]
factor-Rle of length 0 with 0 runs
  Lengths: 
  Values : 
Levels(3): + - *




## Round-trip: R to Python
The true test of a plugin is the ability to bring data back without loss of information.

In [6]:
# R -> Python Conversion
sce_back = exp_engine.r2py(r_sce)

# Verify identity
assert (sce_back.get_assay("counts") == counts).all()
assert sce_back.get_column_names() == ["Cell_1", "Cell_2", "Cell_3", "Cell_4"]
assert sce_back.get_row_ranges().get_seqnames() == ["chr1", "chr1", "chr2", "chrX", "chrY"]

print("Round-trip successful! All slots synchronized.")

<class 'dict'>
<class 'genomicranges.grangeslist.CompressedGenomicRangesList'>


NotImplementedError: Conversion 'py2rpy' not defined for objects of type '<class 'genomicranges.grangeslist.CompressedGenomicRangesList'>'

In [8]:
from bioc2ri.bioc_ranges_plugin import bioc_ranges_plugin
ranges_eng = bioc_ranges_plugin()

r_row_ranges = r["rowRanges"](r_sce)

In [9]:
r_row_ranges

<rpy2.robjects.methods.RS4 object at 0x768412ccbcb0> [25]
R classes: ('CompressedGRangesList',)

In [ ]:
import numpy as np
from summarizedexperiment import SummarizedExperiment
from bioc2ri.summarizedexperiment_plugin import bioc_experiment_plugin
from biocframe import BiocFrame

# 1. Setup Data
counts = np.random.poisson(lam=10, size=(100, 10))

# 2. Initialize ONLY the Experiment Engine
# It will internally call numpy_plugin() and biocpy_plugin() as needed
exp_engine = bioc_experiment_plugin()

# 3. Create SE in Python
coldata = BiocFrame({"condition": ["A"] * 5 + ["B"] * 5}, row_names=[f"Sample{i}" for i in range(10)])
se = SummarizedExperiment(assays={"counts": counts}, column_data=coldata)

# 4. Convert using the standalone engine
r_se = exp_engine.py2r(se)

# 5. Convert back
se_back = exp_engine.r2py(r_se)

# print(f"Successfully converted {type(se_back)} using a standalone engine.")


In [5]:
print(se_back)

class: SummarizedExperiment
dimensions: (100, 10)
assays(1): ['counts']
row_data columns(0): []
row_names(0):  
column_data columns(1): ['condition']
column_names(10): ['Sample0', 'Sample1', 'Sample2', 'Sample3', 'Sample4', 'Sample5', 'Sample6', 'Sample7', 'Sample8', 'Sample9']
metadata(0): 


In [4]:
from bioc2ri.lazy_r_env import get_r_environment
renv = get_r_environment()

se_pkg = renv.importr("SummarizedExperiment")

In [5]:
se_pkg.rowRanges

AttributeError: module 'SummarizedExperiment' has no attribute 'rowRanges'

In [6]:
renv.ro.r["rowRanges"]

<rpy2.robjects.functions.SignatureTranslatedFunction object at 0x70dfcdd46820> [3]
R classes: ('standardGeneric',)